<a href="https://colab.research.google.com/github/sandeshkg/gentut/blob/main/gentut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LLM A Hands on appraoch project Gentut

In [1]:
!nvidia-smi

Sat Aug  8 11:37:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.makedirs('/content/drive/MyDrive/GenTut', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/GenTut/hf_cache'  # cache model weights across sessions

In [10]:
%cd /content/drive/MyDrive/GenTut
!git clone https://github.com/sandeshkg/gentut.git
%cd gentut

/content/drive/MyDrive/GenTut
fatal: destination path 'gentut' already exists and is not an empty directory.
/content/drive/MyDrive/GenTut/gentut


2. Install dependencies

In [4]:
%pip install -q langchain langgraph langchain-huggingface langchain-google-genai \
    transformers accelerate bitsandbytes pydantic streamlit google-generativeai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 121.9 MB/s eta 0:00:00


In [5]:
from dotenv import load_dotenv
import os

load_dotenv('/content/drive/MyDrive/GenTut/.env')

hf_token = os.getenv("HF_TOKEN")
gemini_key = os.getenv("GEMINI_API_KEY")

In [6]:
from huggingface_hub import login
login(token=hf_token)

#import google.generativeai as genai
#genai.configure(api_key=gemini_key)

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
messages = [{"role": "user", "content": "Say hello in one sentence."}]
inputs = tokenizer.apply_chat_template(messages,
                                    return_tensors="pt",
                                    add_generation_prompt=True,
                                    return_dict=True).to(model.device)

output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


user

Say hello in one sentence.assistant

Hello!


Evaluation 1 : Schema Fidelity
Failure with below prompt. Enhance the prompt with example instance and not just schema in subsequent prompt.

In [17]:
from eval_utils import ResultsLogger

logger = ResultsLogger()

In [18]:
from schemas import CognitiveState

def test_skill_identifier(student_message):
    schema_prompt = f"""You are a Skill Identifier agent. Given the student message below,
output ONLY a JSON object with these exact fields: student_id, current_topic, skill_level, misconceptions, hint_stage, mastery_score.

Student message: "{student_message}"
"""
    messages = [{"role": "user", "content": schema_prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=True).to(model.device)
    output = model.generate(**inputs, max_new_tokens=300, max_length=None)
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    try:
        state = CognitiveState.model_validate_json(raw)
        logger.log("skill_identifier", schema_prompt, student_message, raw, parsed_state=state)
    except Exception as e:
        logger.log("skill_identifier", schema_prompt, student_message, raw, error=e)

In [20]:
test_messages = [
    "I don't understand why my for loop never terminates.",
    "What's the difference between a list and a tuple?",
    "My recursive function keeps hitting max recursion depth.",
]

for msg in test_messages:
    test_skill_identifier(msg)

In [21]:
df = logger.to_df()
df[["timestamp", "student_message", "valid", "error"]]

,timestamp,student_message,valid,error
0,2026-08-08 12:12:07,I don't understand why my for loop never termi...,False,1 validation error for CognitiveState\n Inval...
1,2026-08-08 12:12:14,What's the difference between a list and a tuple?,False,1 validation error for CognitiveState\n Inval...
2,2026-08-08 12:12:21,My recursive function keeps hitting max recurs...,False,1 validation error for CognitiveState\n Inval...


In [22]:
print(f"Schema Fidelity: {logger.fidelity_pct('skill_identifier'):.1f}%")

Schema Fidelity: 0.0%


In [25]:
logger.save("day1_schema_fidelity_log.csv")

!git add eval_utils.py day1_schema_fidelity_log.csv
!git commit -m "Day 1: add reusable eval harness + schema fidelity results"
!git push

Saved 3 results to day1_schema_fidelity_log.csv
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/

nothing added to commit but untracked files present (use "git add" to track)
fatal: could not read Username for 'https://github.com': No such device or address
